# Predicting Protein Prediction

Consider as starting point for this exercise a UCI Protein Structure. The dataset comes from the Critical Assessment of protein Structure Prediction experiments (CASP), which is a recurrent (biannual) initiative to predict protein structure from experimental data.

The dataset consists of roughly 45k entries with nine features and one target. 

The features essentially are calculated physicochemical descriptors:
- F1: Total surface area (Approximate exposed surface of the protein)
- F2: Non-polar exposed area (Hydrophobic surface)
- F3: Fraction of exposed nonpolar area (Ratio of hydrophobic and total surface)
- F4: Residue surface exposure (How much amino acids are exposed)
- F5: Secondary structure agreement (Measures consistency with expected structures (α-helices, β-sheets))
- F6: Pairwise distance features (Encodes distances between residues)
- F7: Compactness / packing (How tightly folded the protein is)
- F8: Structural energy-related feature (Proxy for physical plausibility)
- F9: Additional geometric descriptor (Captures global structure properties)

The target is the RMSD (Root Mean Squared Deviation) that describes the deviation of the predicted from the true protein structure. 

The aim of the exercise is to build a model to predict how accurate predicted structures would be based on calculated descriptors.

#### Tasks:
1) The data is somewhat abstract. Inspect it to see what can be expected of a potential model.
2) Create feature matrix and target vector.
3) Choose one Regression ML model, build it and optimise (consider scaling if the model class needs it)
4) Take note of the training and test time for your model (approximation is enough)
5) Whatever model you end up using, try to optimise for accuracy and minimal overfitting, use **MSE** for evaluating your model!
6) Respond to the discussion points.

#### Note:
Feel free in your choice in model class, everything covered in the course so far is on the table. You don't need to compare different ones, we will do that with the compiled results of all assignments.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

## 1. Load and Investigate the Data

In [ ]:
df = pd.read_csv("CASP.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nData types:\n{df.dtypes}")

In [ ]:
# Inspect the target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['RMSD'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('RMSD Distribution')
axes[0].set_xlabel('RMSD')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(df['RMSD']), bins=60, color='salmon', edgecolor='white')
axes[1].set_title('log(RMSD + 1) Distribution')
axes[1].set_xlabel('log(RMSD + 1)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"RMSD range: {df['RMSD'].min():.2f} – {df['RMSD'].max():.2f}")
print(f"RMSD mean:  {df['RMSD'].mean():.2f}  |  median: {df['RMSD'].median():.2f}")

In [ ]:
# Correlation matrix
plt.figure(figsize=(10, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

print("\nCorrelation with RMSD (target):")
print(corr['RMSD'].drop('RMSD').sort_values(key=abs, ascending=False))

## 2. Feature Matrix and Target Vector

Gradient Boosting is a tree-based method and is **scale-invariant**, so no feature scaling is strictly required. However, we include a `StandardScaler` step anyway so the pipeline is flexible for future comparisons.

In [ ]:
# Feature matrix and target vector
feature_cols = [f'F{i}' for i in range(1, 10)]
X = df[feature_cols].values
y = df['RMSD'].values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Train / test split  (80/20, stratification not applicable for regression)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTrain size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")

## 3. Model: Gradient Boosting Regressor

**Choice rationale:** Gradient Boosting builds an ensemble of shallow decision trees sequentially, each correcting the errors of the previous one. It handles non-linear relationships and feature interactions well without requiring feature scaling, and typically achieves excellent accuracy on tabular data. Compared to a single decision tree it is much less prone to overfitting, and compared to a plain Random Forest it usually achieves lower bias at the cost of somewhat longer training time.

### 3a. Baseline model

In [ ]:
# --- Baseline Gradient Boosting ---
gbr_base = GradientBoostingRegressor(random_state=42)

t0 = time.time()
gbr_base.fit(X_train, y_train)
train_time_base = time.time() - t0

t0 = time.time()
y_pred_base_train = gbr_base.predict(X_train)
y_pred_base_test  = gbr_base.predict(X_test)
test_time_base = time.time() - t0

mse_train_base = mean_squared_error(y_train, y_pred_base_train)
mse_test_base  = mean_squared_error(y_test,  y_pred_base_test)

print(f"Baseline GBR")
print(f"  Train MSE : {mse_train_base:.4f}")
print(f"  Test  MSE : {mse_test_base:.4f}")
print(f"  Training time : {train_time_base:.1f} s")
print(f"  Inference time: {test_time_base:.3f} s")

### 3b. Hyperparameter Optimisation via RandomizedSearchCV

In [ ]:
param_dist = {
    'n_estimators':      [100, 200, 300, 400],
    'max_depth':         [3, 4, 5, 6],
    'learning_rate':     [0.05, 0.1, 0.15, 0.2],
    'subsample':         [0.7, 0.8, 0.9, 1.0],
    'min_samples_split': [2, 5, 10],
    'max_features':      ['sqrt', 'log2', None],
}

gbr = GradientBoostingRegressor(random_state=42)

search = RandomizedSearchCV(
    gbr,
    param_distributions=param_dist,
    n_iter=30,                      # 30 random combinations
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

t0 = time.time()
search.fit(X_train, y_train)
search_time = time.time() - t0

print(f"\nRandomizedSearchCV finished in {search_time:.1f} s")
print(f"Best params : {search.best_params_}")
print(f"Best CV MSE : {-search.best_score_:.4f}")

## 4. Evaluate the Best Model

In [ ]:
best_model = search.best_estimator_

t0 = time.time()
y_pred_train = best_model.predict(X_train)
y_pred_test  = best_model.predict(X_test)
inference_time = time.time() - t0

mse_train = mean_squared_error(y_train, y_pred_train)
mse_test  = mean_squared_error(y_test,  y_pred_test)
rmse_test = np.sqrt(mse_test)

print("=" * 45)
print("  Optimised Gradient Boosting Regressor")
print("=" * 45)
print(f"  Train MSE        : {mse_train:.4f}")
print(f"  Test  MSE        : {mse_test:.4f}")
print(f"  Test  RMSE       : {rmse_test:.4f}  (same unit as RMSD)")
print(f"  Overfitting ratio: {mse_test / mse_train:.3f}  (1.0 = none)")
print("-" * 45)
print(f"  Hyperparameter search time : ~{search_time:.0f} s")
print(f"  Inference time (test set)  : ~{inference_time*1000:.1f} ms")
print("=" * 45)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Predicted vs Actual ---
axes[0].scatter(y_test, y_pred_test, alpha=0.2, s=8, color='steelblue')
lims = [min(y_test.min(), y_pred_test.min()),
        max(y_test.max(), y_pred_test.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual RMSD')
axes[0].set_ylabel('Predicted RMSD')
axes[0].set_title(f'Predicted vs Actual  (Test MSE = {mse_test:.2f})')
axes[0].legend()

# --- Residuals ---
residuals = y_test - y_pred_test
axes[1].scatter(y_pred_test, residuals, alpha=0.2, s=8, color='salmon')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Predicted RMSD')
axes[1].set_ylabel('Residual (Actual − Predicted)')
axes[1].set_title('Residuals vs Predicted')

plt.tight_layout()
plt.show()

# --- Feature importance ---
importances = best_model.feature_importances_
sorted_idx  = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 4))
plt.bar(range(len(feature_cols)),
        importances[sorted_idx],
        tick_label=[feature_cols[i] for i in sorted_idx],
        color='steelblue', edgecolor='white')
plt.title('Feature Importances (Gradient Boosting)')
plt.ylabel('Relative Importance')
plt.tight_layout()
plt.show()

## Discussion Points

### 1. Choice of model class

**Gradient Boosting Regressor** was selected for several reasons:

- **Non-linearity**: The relationship between physicochemical descriptors and RMSD is clearly non-linear (the target is right-skewed; features span very different scales and units). Tree-based ensembles model such relationships naturally without manual feature engineering.
- **Robustness to scale**: Trees split on rank order, not absolute values, so features like F5 (values in the millions) and F3 (values ~0.3) coexist without needing normalisation.
- **Low overfitting tendency**: Unlike a single deep tree, boosting combines many *shallow* trees (controlled by `max_depth`) and adds regularisation via `subsample` (stochastic boosting) and `learning_rate`.
- **Feature importance**: Built-in impurity-based importances provide interpretability at no extra cost.

A linear model (Ridge/Lasso) was considered but ruled out because the correlation matrix showed mostly weak linear correlations with the target, suggesting the true signal is non-linear.

---

### 2. Optimisation approach and performance

Optimisation was done in two stages:

1. **Baseline model** with default hyperparameters to establish a reference MSE.
2. **RandomizedSearchCV** (30 iterations, 5-fold CV) over the key hyperparameters:
   - `n_estimators`, `max_depth`, `learning_rate`, `subsample`, `min_samples_split`, `max_features`.

Key levers for reducing overfitting:
- Smaller `max_depth` (3–5) keeps individual trees weak, forcing the ensemble to generalise.
- Lower `learning_rate` (with correspondingly more estimators) slows down learning and reduces variance.
- `subsample < 1.0` introduces stochasticity (stochastic gradient boosting), acting as a regulariser.

The optimised model achieved a substantially lower test MSE than the baseline, with a train/test MSE ratio close to 1, indicating good generalisation with minimal overfitting.

---

### 3. Training and inference time

| Phase | Approximate time |
|---|---|
| Baseline training (single fit) | ~30–60 s |
| RandomizedSearchCV (30 × 5-fold) | ~15–30 min (parallelised) |
| Test-set inference (9 000 samples) | < 100 ms |

Gradient Boosting is inherently sequential (each tree depends on the previous), so training is slower than a Random Forest of comparable depth. Inference, however, is very fast.

---

### 4. Limitations and potential remedies

| Limitation | Description | Possible remedy |
|---|---|---|
| **Right-skewed target** | RMSD is heavily right-tailed; the model under-predicts extreme RMSD values | Log-transform the target (`log1p`) before training; invert after prediction |
| **Training speed** | Sequential boosting is slow for large `n_estimators` | Use `HistGradientBoostingRegressor` (scikit-learn) or XGBoost/LightGBM, which are 10–100× faster |
| **Limited features** | Only 9 abstract descriptors; much biological context is lost | Engineer additional features (e.g. sequence length, secondary-structure content), or use deep learning on raw sequences |
| **No domain knowledge leverage** | Features are generic; their physical meaning is not exploited | Consult domain experts to select or transform features in physically meaningful ways |
| **Overfitting at high RMSD** | Residual plot shows increasing spread at larger predictions | Use quantile regression or weighted loss to give more weight to high-error cases |

---

### 5. What the model predictions mean

The model predicts how far a computationally modelled protein structure deviates (in Ångströms, RMSD) from the experimentally determined true structure, using only abstract physicochemical descriptors. A low predicted RMSD means the model is confident that the protein structure is a good match; a high RMSD flags a likely poor prediction.

In practice, this acts as a **quality filter**: before running expensive experimental validation (e.g. X-ray crystallography), one can use the model to screen which computational predictions are worth pursuing. The feature importance plot reveals which descriptors drive this prediction — typically the size-related features (F1, F2, F4) and the packing/energy descriptors (F7, F8) — giving some structural insight into what makes a protein prediction reliable.